#  Previsão de Aluguel em São Paulo
## Notebook 2 — Feature Engineering e Treinamento do Modelo

Neste notebook vamos transformar as variáveis categóricas, treinar um modelo de Random Forest e avaliar sua performance na previsão do valor de aluguel.

Importações de bibliotecas que serao usadas


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score
import pickle


## 1. Carregando o dataset limpo

In [3]:
df_model = pd.read_csv('../data/imoveis_clean.csv')
print(f'Shape: {df_model.shape}')
df_model.head()

Shape: (11425, 8)


,address,district,area,bedrooms,garage,type,rent,total
0,Rua Herval,Belenzinho,21,1,0,Studio e kitnet,2400,2939
1,Avenida São Miguel,Vila Marieta,15,1,1,Studio e kitnet,1030,1345
2,Rua Oscar Freire,Pinheiros,18,1,0,Apartamento,4000,4661
3,Rua Júlio Sayago,Vila Ré,56,2,2,Casa em condomínio,1750,1954
4,Rua Barata Ribeiro,Bela Vista,19,1,0,Studio e kitnet,4000,4654


## 2. Feature Engineering — Transformando variáveis categóricas

In [4]:
le_district = LabelEncoder()
le_type = LabelEncoder()

df_model['district_enc'] = le_district.fit_transform(df_model['district'])
df_model['type_enc'] = le_type.fit_transform(df_model['type'])

X = df_model[['area', 'bedrooms', 'garage', 'district_enc', 'type_enc']]
y = df_model['rent']

print('Features:', X.columns.tolist())
print('Shape X:', X.shape)
print('Shape y:', y.shape)

Features: ['area', 'bedrooms', 'garage', 'district_enc', 'type_enc']
Shape X: (11425, 5)
Shape y: (11425,)


## 3. Dividindo os dados em treino e teste

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Treino:  {X_train.shape[0]:,} registros')
print(f'Teste:   {X_test.shape[0]:,} registros')

Treino:  9,140 registros
Teste:   2,285 registros


## 4. Treinando o modelo

In [7]:
modelo = RandomForestRegressor(n_estimators=100, random_state=42)
modelo.fit(X_train, y_train)

print('Modelo treinado com sucesso!')

Modelo treinado com sucesso!


## 5. Avaliando o modelo


In [8]:
y_pred = modelo.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'MAE:  R$ {mae:,.0f}')
print(f'R²:   {r2:.4f}')

MAE:  R$ 1,019
R²:   0.5584


### Resultados

| Métrica | Valor | Interpretação |
|---|---|---|
| MAE | R$ 1.019 | erro médio por previsão |
| R² | 0.55 | modelo explica 55% da variação dos preços |

O modelo erra em média **R$ 1.019** pra cima ou pra baixo — para um aluguel mediano de R$ 2.415 isso representa ~42% de erro.

Um bom modelo de imóveis costuma atingir R² acima de **0.75**. Nas próximas etapas vamos melhorar essa performance com ajuste de hiperparâmetros e novas features.

## 6. Salvando o modelo

In [10]:
with open('../data/modelo.pkl', 'wb') as f:
    pickle.dump(modelo, f)

with open('../data/le_district.pkl', 'wb') as f:
    pickle.dump(le_district, f)

with open('../data/le_type.pkl', 'wb') as f:
    pickle.dump(le_type, f)

print('Modelo e encoders salvos em data/')

Modelo e encoders salvos em data/
